## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

This first implementation will use a simple, brute-force type of RAG..

In [1]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

In [2]:
# imports for langchain, plotly and Chroma

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings

In [3]:
# Ollama
from langchain_community.chat_models import ChatOllama
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_core.callbacks import StdOutCallbackHandler

In [4]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [5]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [6]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase

folders = glob.glob("knowledge-base/*")

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

# With thanks to CG and Jon R, students on the course, for this fix needed for some users 
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work, some Windows users might need to uncomment the next line instead
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=300)
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")
print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")

Created a chunk of size 1088, which is longer than the specified 1000


Total number of chunks: 130
Document types found: {'products', 'company', 'contracts', 'employees'}


## A sidenote on Embeddings, and "Auto-Encoding LLMs"

We will be mapping each chunk of text into a Vector that represents the meaning of the text, known as an embedding.

OpenAI offers a model to do this, which we will use by calling their API with some LangChain code.

This model is an example of an "Auto-Encoding LLM" which generates an output given a complete input.
It's different to all the other LLMs we've discussed today, which are known as "Auto-Regressive LLMs", and generate future tokens based only on past context.

Another example of an Auto-Encoding LLMs is BERT from Google. In addition to embedding, Auto-encoding LLMs are often used for classification.

### Sidenote

In week 8 we will return to RAG and vector embeddings, and we will use an open-source vector encoder so that the data never leaves our computer - that's an important consideration when building enterprise systems and the data needs to remain internal.

In [7]:
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk
# Chroma is a popular open source Vector Database based on SQLLite

embeddings = OpenAIEmbeddings() # 1536

# If you would rather use the free Vector Embeddings from HuggingFace sentence-transformers
# Then replace embeddings = OpenAIEmbeddings()
# with:
#from langchain.embeddings import HuggingFaceEmbeddings
#embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # 384
#embeddings = HuggingFaceEmbeddings(model_name='bert-base-nli-mean-tokens') # 768
#embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en") # 1024

# Delete if already exists

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# Create vectorstore

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 130 documents


In [8]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 130 vectors with 1,536 dimensions in the vector store


## Visualizing the Vector Store

Let's take a minute to look at the documents and their embedding vectors to see what's going on.

In [9]:
# Prework (with thanks to Jon R for identifying and fixing a bug in this!)

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [10]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [11]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Time to use LangChain to bring it all together

In [12]:
# create a new Chat with OpenAI
#llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# Alternative - if you'd like to use Ollama locally, uncomment this line instead
llm = ChatOpenAI(temperature=0.7, model_name='llama3.2', base_url='http://localhost:11434/v1', api_key='ollama')

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

C:\Users\ssre_\AppData\Local\Temp\ipykernel_22608\2806724369.py:8: LangChainDeprecationWarning:

Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/



In [13]:
# Let's try a simple question

query = "Please explain what Insurellm is in a couple of sentences"
result = conversation_chain.invoke({"question": query})
print(result["answer"])

I don't know what Insurellm specifically refers to. The provided context mentions an agreement between Insurellm and GreenValley Insurance, but I couldn't identify a widely recognized company or entity by that name. If you could provide more information or context about Insurellm, I may be able to help better.


In [14]:
# set up a new conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

## Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [15]:
# Wrapping that in a function

def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [16]:
# And in Gradio:

view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [ ]:
# Let's investigate what gets sent behind the scenes

# That prints what is going on behind the scenes
from langchain_core.callbacks import StdOutCallbackHandler

llm = ChatOpenAI(temperature=0.7, model_name=MODEL)
#llm = ChatOllama(model="llama3:8b", temperature=0.7)
#llm = ChatOpenAI(temperature=0.7, model_name='llama3:8b', base_url='http://localhost:11434/v1', api_key='ollama')

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

#retriever = vectorstore.as_retriever()
retriever = vectorstore.as_retriever(search_kwargs={"k": 50})

conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()])

#query = "Who received the prestigious IIOTY award in 2023? Please stick only to the information it is provided by LangChain/Chroma."
#query = "what is the Date of Birth of Alex Thomson from insurancellm? Information is in the retriever."
query = "How many employeers are there in insurancellm with name Emily?"
result = conversation_chain.invoke({"question": query})
answer = result["answer"]
print("\nAnswer:", answer)

#Wrong chunks were taken!!! the right context is not provided correctly!



> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
## Compensation History
| Year | Base Salary | Bonus         | Total Compensation |
|------|-------------|---------------|--------------------|
| 2023 | $70,000     | $10,000       | $80,000            |
| 2022 | $65,000     | $8,000        | $73,000            |
| 2021 | $60,000     | $5,000        | $65,000            |

## Other HR Notes
- **Professional Development:** Emily is currently enrolled in a leadership training program to enhance her management skills and aims to move into a senior account role within the next 2 years.  
- **Volunteer Work:** Actively participates in community outreach programs, representing Insurellm in charity even

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains.history_aware_retriever import create_history_aware_retriever
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import StdOutCallbackHandler
import re
from custom_langchain import CustomQARetriever

# 1. Load the local Ollama model (e.g., Qwen3:8b)
llm = ChatOllama(model="qwen3:8b", temperature=0.7)
#llm = ChatOllama(model="llama3:8b", temperature=0.7, callbacks=[StdOutCallbackHandler()])
#llm = ChatOpenAI(temperature=0.7, model_name='qwen3:8b', base_url='http://localhost:11434/v1', api_key='ollama')

# 2. Set up memory
memory = ConversationBufferMemory(return_messages=True)

# 3. Build retriever (adjust path if needed)
#vectorstore = Chroma(persist_directory="your_vectorstore_path", embedding_function=your_embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 25})

# 4. Prompt to rephrase question using chat history
retriever_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that rephrases questions for retrieval."),
    ("human", "{chat_history}\nQuestion: {input}")
])

# 5. Create history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retriever,
    prompt=retriever_prompt
)

# 6. Prompt to generate answers from documents
#qa_prompt = ChatPromptTemplate.from_messages([
#    ("system", "You are a helpful assistant. Use the retrieved context to answer the question."),
#    ("human", "Context:\n{context}\n\nQuestion: {input}")
#])
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are an assistant answering questions based ONLY on the provided context. "
     "If the answer is not in the context, say 'I don't know.'"
     "Do NOT include any internal thoughts or tags like <think>. "
     "Give only the final answer as a direct response."
     ),
    ("human", 
     "Context:\n{context}\n\nQuestion: {input}\n\nRemember, do not use prior knowledge. "
     "Do not answer starting with 'Based on the provided context'. "
     "Look at carefully the sections, for example, questions related to employees are likely to be answered from chunks belonging to the 'employees' folder."
     ),
])

# 7. Create document chain (stuffing retrieved docs into prompt)
document_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

# 8. Final retrieval QA chain
retrieval_chain = create_retrieval_chain(
    retriever=history_aware_retriever,
    combine_docs_chain=document_chain,
    #callbacks=[StdOutCallbackHandler()]
)

def clean_answer(answer):
    return re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()

# 9. Ask a question
#query = "What is the date of birth of Avery from insurancellm?"
#query = "How many employeers are there in insurancellm with name Emily?"
query = "There are two Emily names in the company. Can you confirm it?"
#query = "what is insurancellm?"
#query = "Is Avery Lancaster the CEO of Insurancellm?"
#query = "can you describe briefly one product of the company?"
query = "who is junger than 18 years old?"
result = retrieval_chain.invoke({"input": query, "chat_history": []})  # Use memory.messages for ongoing chats
result = clean_answer(result["answer"])
print("\nAnswer:", result)


TypeError: create_retrieval_chain() got an unexpected keyword argument 'callbacks'

In [12]:
!pip install -U langchain_ollama

  Attempting uninstall: ollama
    Found existing installation: ollama 0.4.7
    Uninstalling ollama-0.4.7:
      Successfully uninstalled ollama-0.4.7


In [17]:
from custom_langchain import CustomQARetriever

extra_context = "Look at carefully the sections, for example, questions related to employees are likely to be answered from chunks belonging to the 'employees' folder."
qa_system = CustomQARetriever(
    vectorstore=vectorstore,
    model_name="qwen3:8b",
    temperature=0.7,
    k=25,
    extra_commands=extra_context,
    )

question = "Is Avery Lancaster the CEO of Insurellm?"
question = "How many names 'Emily' does the company have?"
chat_history = []

response = qa_system.answer_question(question, chat_history)
print(response)

The company has two employees named Emily: Emily Carter and Emily Tran.


In [55]:
# create a new Chat with OpenAI
#llm = ChatOpenAI(temperature=0.7, model_name=MODEL)
#llm = ChatOpenAI(temperature=0.7, model_name='llama3.2', base_url='http://localhost:11434/v1', api_key='ollama')
#llm = ChatOllama(model="qwen3:8b", temperature=0.7)
llm = ChatOpenAI(temperature=0.7, model_name='qwen3:8b', base_url='http://localhost:11434/v1', api_key='ollama')

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG; k is how many chunks to use
# 25 nearest chunks!
retriever = vectorstore.as_retriever(search_kwargs={"k": 50})

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [102]:
def chat(question, history):
    #result = conversation_chain.invoke({"question": question})
    result = retrieval_chain.invoke({"input": question, "chat_history": history})
    return result["answer"]

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7875

To create a public link, set `share=True` in `launch()`.


Number of requested results 100 is greater than number of elements in index 66, updating n_results = 66
Number of requested results 100 is greater than number of elements in index 66, updating n_results = 66


# Exercises

Try applying this to your own folder of data, so that you create a personal knowledge worker, an expert on your own information!